<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"></ul></div>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("06-API.DataFrames-SQL")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/09/24 12:27:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 12:27:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/24 12:27:39 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20260924122738-0027/events_1_app-20260924122738-0027.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20260924122738-0027
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

In [4]:
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType

meteoDataFrame  = spark.read.format('csv')\
    .option('sep',';')\
    .option('header','true')\
    .option('nullValue','mq')\
    .option('inferSchema', 'true')\
    .load('../data/meteo/')\
    .cache()

schema = StructType([
        StructField('Id'           , StringType() , True),
        StructField('ville'        , StringType() , True),
        StructField('latitude'     , FloatType() , True),
        StructField('longitude'    , FloatType() , True),
        StructField('altitude'     , IntegerType() , True)])

villes  = spark.read.format('csv')   \
      .option('sep',';')                \
      .option('mergeSchema', 'true')    \
      .option('header','true')          \
      .schema(schema)                   \
      .load('../data/postesSynop.csv')  \
      .cache()

@udf("string")
def formatVille(ville):
    if ville in ['CLERMONT-FD','MONT-DE-MARSAN',
                                   'ST-PIERRE','ST-BARTHELEMY METEO'] :
        return ville.title()
    else :
        if ville.find('-') != -1 :
            return ville[0:ville.find('-')].title()
        else:
            return ville.title()

villesT  = villes.select(
                col('Id').alias('id'),
                formatVille('ville').alias('ville'),
               'latitude',
               'longitude',
               'altitude')


meteo = meteoDataFrame.select(
                 col('numer_sta'),
                 col('date')[0:4].cast('int') ,
                 col('date')[5:2].cast('int'),
                 col('date')[7:2].cast('int'),
                 col('date')[5:4],
                 round(col('t') - 273.15,2),
                 col('u') / 100 ,
                 col('vv') / 1000 ,
                 col('pres') / 1000,
                 coalesce( col('rr3'),
                           col('rr24')/8,
                           col('rr12')/4,
                           col('rr6')/2,
                           col('rr1')*3  ) )\
             .toDF('id','annee','mois','jour','mois_jour','temperature',
                   'humidite','visibilite','pression','precipitations')\
             .cache()

meteo.select('annee','mois','jour','temperature','humidite',
             'visibilite','pression').toPandas().head(5)

26/09/24 12:27:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

,annee,mois,jour,temperature,humidite,visibilite,pression
0,2024,5,1,13.1,0.94,14.20,100.13
1,2024,5,1,14.5,0.85,16.03,100.56
2,2024,5,1,8.8,0.88,NaN,100.74
3,2024,5,1,9.2,0.91,11.05,100.14
4,2024,5,1,12.6,0.97,14.33,99.08


In [5]:
# meteo.write.mode("overwrite").parquet("s3a://lakehouse/bronze/meteoFrance")

In [6]:
meteo.write\
       .mode('overwrite')\
       .format('parquet')\
       .partitionBy('annee')\
       .option('path', 's3a://lakehouse/bronze/meteoFrance')\
       .save()

In [7]:
spark.sql("select * from parquet."+
          "`s3a://lakehouse/bronze/meteoFrance` "+
          "where annee = 2023").toPandas().head(5)

,id,mois,jour,mois_jour,temperature,humidite,visibilite,pression,precipitations,annee
0,7005,8,1,0801,15.4,0.97,20.00,99.46,0.4,2023
1,7015,8,1,0801,15.6,0.95,51.23,99.63,0.6,2023
2,7020,8,1,0801,16.3,0.92,12.00,100.43,0.0,2023
3,7027,8,1,0801,15.9,0.93,49.78,99.81,2.0,2023
4,7037,8,1,0801,15.6,0.98,3.18,98.61,13.4,2023


In [8]:
meteoFance = spark.read.format('parquet').load('s3a://lakehouse/bronze/meteoFrance')

In [9]:
spark.sql('DROP DATABASE IF EXISTS cours CASCADE').toPandas().head()

""


In [10]:
spark.sql('CREATE DATABASE cours').toPandas().head()

""


In [11]:
spark.sql('show databases').toPandas()

,namespace
0,cours
1,default


In [12]:
meteo=spark.sql("""
                SELECT * 
                FROM parquet.`s3a://lakehouse/bronze/meteoFrance`""") 

In [13]:
meteo.toPandas().head()

,id,mois,jour,mois_jour,temperature,humidite,visibilite,pression,precipitations,annee
0,7005,5,1,0501,13.1,0.94,14.20,100.13,0.2,2024
1,7015,5,1,0501,14.5,0.85,16.03,100.56,-0.1,2024
2,7020,5,1,0501,8.8,0.88,NaN,100.74,0.0,2024
3,7027,5,1,0501,9.2,0.91,11.05,100.14,0.0,2024
4,7037,5,1,0501,12.6,0.97,14.33,99.08,0.0,2024


In [14]:
spark.sql('USE cours').toPandas()

""


In [15]:
spark.sql('show tables').toPandas()

,namespace,tableName,isTemporary


In [16]:
chemin = "s3a://lakehouse/spark-warehouse/cours.db/meteo"

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
jpath = spark._jvm.org.apache.hadoop.fs.Path(chemin)
filesystem = jpath.getFileSystem(hadoop_conf)

# Vérifier le chemin avant suppression
print("Existe :", filesystem.exists(jpath))
print("Chemin :", jpath.toString())

Existe : False
Chemin : s3a://lakehouse/spark-warehouse/cours.db/meteo


In [17]:
supprime = filesystem.delete(jpath, True)  # True = suppression récursive
print("Suppression effectuée :", supprime)

Suppression effectuée : False


In [18]:
spark.sql('USE cours').toPandas()
meteo.write.saveAsTable(name="meteo", mode="overwrite")

In [19]:
spark.sql('show tables').toPandas()

,namespace,tableName,isTemporary
0,cours,meteo,False


In [20]:
spark.sql("""SELECT *
            FROM meteo
            WHERE ANNEE = 2024""").toPandas().head(10)

,id,mois,jour,mois_jour,temperature,humidite,visibilite,pression,precipitations,annee
0,7005,5,1,0501,13.1,0.94,14.20,100.13,0.2,2024
1,7015,5,1,0501,14.5,0.85,16.03,100.56,-0.1,2024
2,7020,5,1,0501,8.8,0.88,NaN,100.74,0.0,2024
3,7027,5,1,0501,9.2,0.91,11.05,100.14,0.0,2024
4,7037,5,1,0501,12.6,0.97,14.33,99.08,0.0,2024
5,7072,5,1,0501,12.8,0.93,18.39,100.01,1.2,2024
6,7110,5,1,0501,7.8,0.94,33.68,99.62,-0.1,2024
7,7117,5,1,0501,8.5,0.92,NaN,100.11,-0.1,2024
8,7130,5,1,0501,10.2,0.90,19.73,100.50,0.2,2024
9,7139,5,1,0501,10.1,0.95,20.00,99.18,0.0,2024


In [21]:
spark.sql("""SELECT annee, 
                    avg(temperature) temperature, 
                    avg(humidite) humidite, 
                    avg(visibilite) visibilite, 
                    avg(pression) pression
            FROM meteo
            GROUP BY ANNEE""").toPandas().head(20)

,annee,temperature,humidite,visibilite,pression
0,2023,16.150763,0.756816,26.414694,99.975505
1,2025,15.944294,0.760353,27.219698,99.960841
2,2024,15.698337,0.780930,25.625652,99.954934


In [22]:
spark.sql('USE cours').toPandas()

""


In [23]:
spark.sql('show tables').toPandas()

,namespace,tableName,isTemporary
0,cours,meteo,False


In [24]:
spark.sql("DROP TABLE IF EXISTS cours.meteo PURGE")

DataFrame[]